# Busca de hiperparâmetros (classificação multiclasse)

Este notebook testa algumas arquiteturas e combinações de parâmetros com o Optuna.
A validação ajuda a escolher os parâmetros; o conjunto de teste fica guardado para uma avaliação final.

In [1]:
# Importações usadas no preparo dos dados e no tuning
import os
import json
import numpy as np
import pandas as pd

# Bibliotecas de gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch e Optuna para o Tuning (obs: n consegui fazer o tensorflow reconhecer minha gpu)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import optuna

# Utilitários de Machine Learning do Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# Usa uma semente fixa para facilitar a repetição dos testes e reduz as mensagens do Optuna
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Usa a GPU quando disponível; se não, o código roda na CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")

Usando dispositivo: cuda


In [2]:
# Lê os dados. A coluna label é a resposta que o modelo vai tentar prever.
df = pd.read_csv("Normal+Attacks.csv")

print(f"Dimensões do novo dataset: {df.shape[0]} linhas e {df.shape[1]} colunas")

# Separa as entradas do rótulo e mantém só as colunas numéricas que estas redes usam.
X = df.drop(columns=["label"], errors="ignore")
X = X.select_dtypes(include="number")

# Definição do cenário pro notebook
CENARIO = "multiclasse"

# No binário, Normal vira 0 e qualquer ataque vira 1; no multiclasse cada classe recebe um número.
if CENARIO == "binario":
    y = df["label"].apply(lambda x: 0 if x.strip().lower() == "normal" else 1)
    n_classes = 2
else:
    le = LabelEncoder()
    y = le.fit_transform(df["label"])
    n_classes = len(le.classes_)

# Guarda 25% para o teste final. O tuning não usa esse pedaço.
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

# Mantém os rótulos em um array para separar treino e validação depois.
y_train = np.asarray(y_train_raw)

# Número de colunas usadas como entrada das redes.
N_FEATURES = X_train_raw.shape[1]

print(f"\nDados prontos para o cenário: {CENARIO.upper()}")
print(f"Número de colunas numéricas de entrada (N_FEATURES): {N_FEATURES}")
print(f"Número de classes identificadas: {n_classes}")


Dimensões do novo dataset: 149434 linhas e 45 colunas

Dados prontos para o cenário: MULTICLASSE
Número de colunas numéricas de entrada (N_FEATURES): 44
Número de classes identificadas: 10


In [3]:
# As redes recebem as mesmas entradas, mas organizadas de formas diferentes.
# Nas recorrentes e na CNN, cada coluna é tratada como um passo da sequência.

# MLP: recebe todas as colunas de uma vez.
class MLP_Tuning(nn.Module):
    def __init__(self, input_dim, n_camadas, unidades, dropouts, ativacao, n_classes):
        super().__init__()
        camadas = []
        prev = input_dim
        for i in range(n_camadas):
            camadas.append(nn.Linear(prev, unidades[i]))
            camadas.append(nn.ReLU() if ativacao == "relu" else nn.Tanh())
            if dropouts[i] > 0: camadas.append(nn.Dropout(dropouts[i]))
            prev = unidades[i]
        camadas.append(nn.Linear(prev, 1 if n_classes == 2 else n_classes))
        self.rede = nn.Sequential(*camadas)
    def forward(self, x): return self.rede(x)

# RNN, GRU e LSTM usam a mesma estrutura, mudando o tipo de célula.
class Recorrente_Tuning(nn.Module):
    def __init__(self, tipo, input_dim, n_camadas, unidades, dropouts, n_classes):
        super().__init__()
        self.tipo = tipo
        self.camadas = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        prev = input_dim
        for i in range(n_camadas):
            if tipo == "RNN": camada = nn.RNN(prev, unidades[i], batch_first=True)
            elif tipo == "GRU": camada = nn.GRU(prev, unidades[i], batch_first=True)
            else: camada = nn.LSTM(prev, unidades[i], batch_first=True)
            self.camadas.append(camada)
            self.dropouts.append(nn.Dropout(dropouts[i]) if dropouts[i] > 0 else nn.Identity())
            prev = unidades[i]
        self.densa = nn.Linear(prev, 64)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(64, 1 if n_classes == 2 else n_classes)
    def forward(self, x):
        for i, (camada, drop) in enumerate(zip(self.camadas, self.dropouts)):
            out, _ = camada(x)
            out = drop(out)
            if i < len(self.camadas) - 1: x = out
            else: x = out[:, -1, :] # Pega apenas o último passo temporal
        return self.saida(self.relu(self.densa(x)))

# CNN: aplica filtros ao longo das colunas de entrada.
class CNN_Tuning(nn.Module):
    def __init__(self, input_channels, n_blocos, filtros, kernel_size, n_classes):
        super().__init__()
        camadas = []
        prev = input_channels
        for i in range(n_blocos):
            camadas.append(nn.Conv1d(prev, filtros[i], kernel_size=kernel_size, padding=kernel_size//2))
            camadas.append(nn.ReLU())
            camadas.append(nn.MaxPool1d(kernel_size=2, ceil_mode=True))
            prev = filtros[i]
        self.conv = nn.Sequential(*camadas)
        self.pool_global = nn.AdaptiveAvgPool1d(1)
        self.densa = nn.Linear(prev, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.saida = nn.Linear(128, 1 if n_classes == 2 else n_classes)
    def forward(self, x):
        x = self.conv(x)
        x = self.pool_global(x).squeeze(-1)
        return self.saida(self.dropout(self.relu(self.densa(x))))


In [4]:
# O Optuna testa combinações de parâmetros e guarda os melhores resultados.
DIR_TUNING = "tuning_bayesiano"
os.makedirs(DIR_TUNING, exist_ok=True)

MAX_TRIALS = 25
EPOCHS_BUSCA = 25
BATCH_SIZE = 128

# Separa 10% do treino para validação, mantendo a proporção das classes.
X_fit_raw, X_val_raw, y_fit, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.1, random_state=SEED, stratify=y_train
)

# O scaler aprende só com os dados usados para ajustar os modelos deste tuning.
scaler = StandardScaler()
X_fit = scaler.fit_transform(X_fit_raw)
X_val = scaler.transform(X_val_raw)

if CENARIO == "multiclasse":
    y_fit_tensor = torch.tensor(y_fit, dtype=torch.long)
    y_val_tensor = torch.tensor(y_val, dtype=torch.long)
else:
    y_fit_tensor = torch.tensor(y_fit, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

# Divide os dados em lotes; só os lotes de treino são embaralhados.
train_dataset = TensorDataset(torch.tensor(X_fit, dtype=torch.float32), y_fit_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)

def criar_objetivo(nome_modelo):
    def objetivo(trial):
        # Cada tentativa escolhe uma taxa de aprendizado e uma arquitetura.
        lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
        
        if nome_modelo == "MLP":
            n_camadas = trial.suggest_int("n_camadas", 1, 3)
            unidades = [trial.suggest_int(f"unidades_{i}", 32, 256, step=32) for i in range(n_camadas)]
            dropouts = [trial.suggest_float(f"dropout_{i}", 0.0, 0.5, step=0.1) for i in range(n_camadas)]
            ativacao = trial.suggest_categorical("ativacao", ["relu", "tanh"])
            model = MLP_Tuning(N_FEATURES, n_camadas, unidades, dropouts, ativacao, n_classes).to(DEVICE)
        
        elif nome_modelo in ["RNN", "GRU", "LSTM"]:
            n_camadas = trial.suggest_int("n_camadas", 1, 2)
            unidades = [trial.suggest_int(f"unidades_{i}", 16, 128, step=16) for i in range(n_camadas)]
            dropouts = [trial.suggest_float(f"dropout_{i}", 0.0, 0.4, step=0.1) for i in range(n_camadas)]
            model = Recorrente_Tuning(nome_modelo, 1, n_camadas, unidades, dropouts, n_classes).to(DEVICE)
            
        elif nome_modelo == "CNN":
            n_blocos = trial.suggest_int("n_blocos_conv", 1, 3)
            filtros = [trial.suggest_int(f"filtros_{i}", 16, 128, step=16) for i in range(n_blocos)]
            kernel_size = trial.suggest_categorical("kernel_size", [3, 5, 7])
            model = CNN_Tuning(1, n_blocos, filtros, kernel_size, n_classes).to(DEVICE)

        # A perda muda conforme o formato dos rótulos do cenário.
        if CENARIO == "multiclasse":
            criterion = nn.CrossEntropyLoss()
        else:
            criterion = nn.BCEWithLogitsLoss()
            
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        # Treina cada configuração por algumas épocas antes de medir a validação.
        for epoch in range(EPOCHS_BUSCA):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                if nome_modelo in ["RNN", "GRU", "LSTM"]: xb = xb.unsqueeze(-1)
                elif nome_modelo == "CNN": xb = xb.unsqueeze(-1).permute(0, 2, 1)
                
                optimizer.zero_grad()
                logits = model(xb)
                
                # Ajusta as dimensões da saída para casar com o critério escolhido
                if CENARIO == "binario":
                    loss = criterion(logits.view(-1), yb)
                else:
                    loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()

        # Mede a validação
        model.eval()
        total_certos = 0
        total_amostras = 0
        with torch.no_grad():
            for xb_v, yb_v in val_loader:
                xb_v, yb_v = xb_v.to(DEVICE), yb_v.to(DEVICE)
                if nome_modelo in ["RNN", "GRU", "LSTM"]: xb_v = xb_v.unsqueeze(-1)
                elif nome_modelo == "CNN": xb_v = xb_v.unsqueeze(-1).permute(0, 2, 1)
                
                logits_v = model(xb_v)
                
                # Cálculo de previsões dinâmico para múltiplas classes
                if CENARIO == "binario":
                    preds = torch.sigmoid(logits_v).view(-1)
                    preds_finais = (preds > 0.5).float()
                else:
                    # No multiclasse, pegamos o índice da maior nota (argmax)
                    preds_finais = torch.argmax(logits_v, dim=1)

                
                total_certos += (preds_finais == yb_v).sum().item()
                total_amostras += yb_v.size(0)
                
        return total_certos / total_amostras

    return objetivo

# --- LOOP PRINCIPAL DE EXECUÇÃO ---
resumo_geral = {}

for nome_modelo in ["MLP", "CNN", "RNN", "GRU", "LSTM"]:
    print("\n" + "=" * 60)
    print(f"Iniciando tuning de {nome_modelo} em {DEVICE}")
    print("=" * 60)
    
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(criar_objetivo(nome_modelo), n_trials=MAX_TRIALS)
    
    print(f"Melhor Acurácia para {nome_modelo}: {study.best_value:.4f}")
    
    resumo_geral[nome_modelo] = {
        "val_accuracy": study.best_value,
        "hiperparametros": study.best_params
    }
    
    with open(os.path.join(DIR_TUNING, f"melhores_hp_{nome_modelo}_{CENARIO}.json"), "w") as f:
        json.dump(resumo_geral[nome_modelo], f, indent=2)

caminho_resumo = os.path.join(DIR_TUNING, f"resumo_melhores_hp_{CENARIO}.json")
with open(caminho_resumo, "w") as f:
    json.dump(resumo_geral, f, indent=2)

print("\n" + "=" * 60)
print(f"TUNING CONCLUÍDO SEM ERROS! ARQUIVO EM: {caminho_resumo}")
print("=" * 60)



Iniciando tuning de MLP em cuda
Melhor Acurácia para MLP: 0.5882

Iniciando tuning de CNN em cuda
Melhor Acurácia para CNN: 0.5911

Iniciando tuning de RNN em cuda
Melhor Acurácia para RNN: 0.5879

Iniciando tuning de GRU em cuda
Melhor Acurácia para GRU: 0.5905

Iniciando tuning de LSTM em cuda
Melhor Acurácia para LSTM: 0.5879

TUNING CONCLUÍDO SEM ERROS! ARQUIVO EM: tuning_bayesiano/resumo_melhores_hp_multiclasse.json
